# The Price is Right

## Week 8 Order of Play

Day 1: Modal.com and SpecialistAgent  
Day 2: RAG, FrontierAgent, Ensemble Agent  
Day 3: ScannerAgent, MessengerAgent  
Day 4: AutonomousPlannerAgent and DealAgentFramework  
Day 5: The Price Is Right Finale


Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection
import logging
import requests
load_dotenv(override=True)
openai = OpenAI()
MODEL = 'gpt-5-mini'

In [3]:
deals = ScrapedDeal.fetch(show_progress=True)

100%|██████████| 3/3 [01:16<00:00, 25.42s/it]


In [4]:
len(deals)

30

In [5]:
deals[10].describe()

'Title: Samsung Galaxy Tab A9+ Tablets at Best Buy: Up to $70 off + free shipping\nDetails: Shop the Galaxy Tab A9+ 11" WiFi tablets from $160. Both 64GB and 128GB options are on sale. Shop Now at Best Buy\nFeatures: \nURL: https://www.dealnews.com/Samsung-Galaxy-Tab-A9-Tablets-at-Best-Buy-Up-to-70-off-free-shipping/21815520.html?iref=rss-c39'

### We are going to ask GPT-5-mini to summarize deals and identify their price

In [6]:
SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 
"""

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""

USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."

In [7]:
# this makes a suitable user prompt given scraped deals

def make_user_prompt(scraped):
    user_prompt = USER_PROMPT_PREFIX
    user_prompt += '\n\n'.join([scrape.describe() for scrape in scraped])
    user_prompt += USER_PROMPT_SUFFIX
    return user_prompt

In [8]:
# Let's create a user prompt for the deals we just scraped, and look at how it begins

user_prompt = make_user_prompt(deals)
print(user_prompt[:2000])
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}]

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: UDPower Big Spring Sale: Up to $1,399 off + free shipping
Details: Save up to $1,399 across a wide range of solar generators, portable battery backups, solar panels, and more. We've pictured the UDPower C600 Portable Power Station for $289.99 - a $69 savings. Plus, all orders ship for free, and UDPower plants a tree for every order. Shop Now at UDPOWER
Features: 
URL: https://w

In [9]:
response = openai.chat.completions.parse(model=MODEL, messages=messages, response_format=DealSelection, reasoning_effort="minimal")
results = response.choices[0].message.parsed
results

DealSelection(deals=[Deal(product_description='EcoFlow River 2 Pro is a portable power station with 768Wh capacity designed for home backup and outdoor use. It includes four 800W AC outlets for powering multiple appliances simultaneously, a USB‑C port, three USB‑A ports, two DC outputs and a car outlet. The unit features advanced battery management system (BMS) protection for safety and reliable performance, and model EFR620 supports higher-power draws for a variety of devices.', price=315.0, url='https://www.dealnews.com/products/Eco-Flow/Eco-Flow-River-2-Pro-768-Wh-Portable-Power-Station/420365.html?iref=rss-c142'), Deal(product_description='Gendome Starlink Mini Battery is a 288Wh LiFePO4 portable battery designed to power Starlink terminals and other devices. It features an integrated power-and-stand design for easy setup, supports dual charging at up to 120W, and includes a smart BMS with LED indicators to monitor status. The package comes with a travel case and accessories for tr

In [10]:
for deal in results.deals:
    print(deal.product_description)
    print(deal.price)
    print(deal.url)
    print()


EcoFlow River 2 Pro is a portable power station with 768Wh capacity designed for home backup and outdoor use. It includes four 800W AC outlets for powering multiple appliances simultaneously, a USB‑C port, three USB‑A ports, two DC outputs and a car outlet. The unit features advanced battery management system (BMS) protection for safety and reliable performance, and model EFR620 supports higher-power draws for a variety of devices.
315.0
https://www.dealnews.com/products/Eco-Flow/Eco-Flow-River-2-Pro-768-Wh-Portable-Power-Station/420365.html?iref=rss-c142

Gendome Starlink Mini Battery is a 288Wh LiFePO4 portable battery designed to power Starlink terminals and other devices. It features an integrated power-and-stand design for easy setup, supports dual charging at up to 120W, and includes a smart BMS with LED indicators to monitor status. The package comes with a travel case and accessories for transport and field use.
200.0
https://www.dealnews.com/Gendome-288-Wh-Starlink-Mini-Batter

In [11]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [12]:
from agents.scanner_agent import ScannerAgent

In [13]:
agent = ScannerAgent()
result = agent.scan()

INFO:root:[Scanner Agent] Scanner Agent is initializing
INFO:root:[Scanner Agent] Scanner Agent is ready
INFO:root:[Scanner Agent] Scanner Agent is about to fetch deals from RSS feed
INFO:root:[Scanner Agent] Scanner Agent received 30 deals not already scraped
INFO:root:[Scanner Agent] Scanner Agent is calling OpenAI using Structured Outputs
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:[Scanner Agent] Scanner Agent received 5 selected deals with price>0 from OpenAI


In [14]:
for deal in results.deals:
    print(deal.product_description)
    print(deal.price)
    print(deal.url)
    print()


EcoFlow River 2 Pro is a portable power station with 768Wh capacity designed for home backup and outdoor use. It includes four 800W AC outlets for powering multiple appliances simultaneously, a USB‑C port, three USB‑A ports, two DC outputs and a car outlet. The unit features advanced battery management system (BMS) protection for safety and reliable performance, and model EFR620 supports higher-power draws for a variety of devices.
315.0
https://www.dealnews.com/products/Eco-Flow/Eco-Flow-River-2-Pro-768-Wh-Portable-Power-Station/420365.html?iref=rss-c142

Gendome Starlink Mini Battery is a 288Wh LiFePO4 portable battery designed to power Starlink terminals and other devices. It features an integrated power-and-stand design for easy setup, supports dual charging at up to 120W, and includes a smart BMS with LED indicators to monitor status. The package comes with a travel case and accessories for transport and field use.
200.0
https://www.dealnews.com/Gendome-288-Wh-Starlink-Mini-Batter

### Introducing Pushover

Pushover is a nifty tool for sending Push Notifications to your phone.

It's super easy to set up and install!

Simply visit https://pushover.net/ and click 'Login or Signup' on the top right to sign up for a free account, and create your API keys.

Once you've signed up, on the home screen, click "Create an Application/API Token", and give it any name (like AIEngineer) and click Create Application.

Then add 2 lines to your `.env` file:

PUSHOVER_USER=_put the key that's on the top right of your Pushover home screen and probably starts with a u_  
PUSHOVER_TOKEN=_put the key when you click into your new application called Agents (or whatever) and probably starts with an a_

Remember to save your `.env` file, and run `load_dotenv(override=True)` after saving, to set your environment variables.

Finally, click "Add Phone, Tablet or Desktop" to install on your phone.

In [22]:
load_dotenv(override=True)

True

In [23]:
pushover_user = os.getenv('PUSHOVER_USER')
pushover_token = os.getenv('PUSHOVER_TOKEN')
pushover_url = "https://api.pushover.net/1/messages.json"

In [24]:
if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

Pushover user found and starts with u
Pushover token found and starts with a


In [ ]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [27]:
push("Hi avigail!!")

Push: Hi avigail!!


In [1]:
from agents.messaging_agent import MessagingAgent

agent = MessagingAgent()
agent.push("SUCH A MASSIVE DEAL!!")

In [2]:
agent.notify("A special deal on Sumsung 60 inch LED TV going at a great bargain", 300, 1000, "www.samsung.com")